In [39]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import gcamreader
import plotly.express as px

In [2]:
# ----------------------------
# Config
# ----------------------------
DBPATH    = "../output/"
DBFILE    = "database_basexdb_korea_2035"
SCENARIOS = ["Current-Policies-Med", "Enhanced-Ambition-Med"]

# Service subsectors to include (passenger + road freight core groups)
SERVICE_SECTORS_ALL   = ["Bus", "Car", "Large Car and Truck", "Medium truck"]
UNITS_PASSENGER       = "million pass-km"
UNITS_FREIGHT         = "million ton-km"

# Colors per scenario
COLOR_MAP = {
    SCENARIOS[0]:      "#636EFA",   # blue
    SCENARIOS[1]:   "#00CC96",   # green
}

# Nicely formatted scenario names for legends
PRETTY = {
    SCENARIOS[0]: "Current Policies",
    SCENARIOS[1]: "Enhanced Ambition",
}

In [3]:
# ----------------------------
# Helpers
# ----------------------------
def connect_and_load_queries():
    conn = gcamreader.LocalDBConn(DBPATH, DBFILE)
    queries = gcamreader.parse_batch_query(os.path.join("..", "output", "queries", "Main_queries.xml"))
    return conn, queries

def base_layout(fig, y2_title, width=600, height=450, legend_pos="h"):
    """Apply consistent layout across figures."""
    # legend defaults
    if legend_pos == "h":
        legend_cfg = dict(x=0.5, y=-0.2, xanchor="center", orientation="h")
    else:
        legend_cfg = dict(
            x=0.02, y=0.98, xanchor="left", yanchor="top",
            bgcolor="rgba(255,255,255,0.6)", bordercolor="lightgray", borderwidth=1,
            orientation="v"
        )

    fig.update_layout(
        barmode="group",
        legend=legend_cfg,
        plot_bgcolor="white",
        paper_bgcolor="white",
        height=height,
        width=width,
        font=dict(size=16)
    )

    fig.update_layout(
        xaxis=dict(
            title="Year",
            showticklabels=True,
            ticks="outside",
            ticklen=6, tickwidth=2, tickcolor="black",
            showline=True, linecolor="black", linewidth=1,
            mirror=True
        ),
        yaxis=dict(
            title=dict(text="New Sales Share (%)", font=dict(color="black")),
            showticklabels=True,
            ticks="outside",
            ticklen=6, tickwidth=2, tickcolor="black",
            showline=True, linecolor="black", linewidth=1,
            dtick=10, mirror=True
        ),
        yaxis2=dict(
            title=dict(text=y2_title, font=dict(color="black")),
            tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
            overlaying="y", side="right",
            showline=True, linecolor="black", linewidth=1
        )
    )
    return fig

In [4]:
# --- New: dedicated runners for the two query types ---
def run_service_query(conn, q, scenarios=SCENARIOS, region="South Korea"):
    """Q159: service output by tech/subsector. We'll later sum and convert to billions."""
    df = conn.runQuery(q, scenarios=scenarios, regions=[region])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def run_share_query(conn, q, scenarios=SCENARIOS, region="South Korea"):
    """Q161: new-vehicle shares (already shares). Normalize to percent if needed."""
    df = conn.runQuery(q, scenarios=scenarios, regions=[region])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    # Heuristic: if shares look like 0-1, convert to %
    if df["value"].dropna().between(0, 1).all():
        df["share_percent"] = df["value"] * 100.0
    else:
        df["share_percent"] = df["value"]  # already in %
    return df

# --- Helpers updated to use separate DFs ---
def aggregate_service(df, service_sectors, min_year=2020):
    """Sum service output across selected subsectors; return totals in billions."""
    sub = df[df["subsector"].isin(service_sectors) & (df['technology'].isin(['BEV', 'FCEV'])) & (df["Year"] >= min_year)].copy()
    out = (sub.groupby(["scenario", "Year", "Units"], as_index=False)["value"]
              .sum().rename(columns={"value": "service"}))
    out["service_billion"] = out["service"] / 1000.0
    return out

def add_dual_series_from_split(fig, df_share, df_service, units_filter, scenario, left_name, right_name):
    """Left: shares from Q161; Right: service from Q159."""
    pretty = PRETTY.get(scenario, scenario)
    clr    = COLOR_MAP.get(scenario, "grey")

    s_share   = df_share[(df_share["scenario"] == scenario) & (df_share["Units"] == units_filter)]
    s_service = df_service[(df_service["scenario"] == scenario) & (df_service["Units"] == units_filter)]

    # y1: new-vehicle share (%)
    fig.add_trace(go.Scatter(
        x=s_share["Year"], y=s_share["share_percent"],
        name=f"{left_name} in {pretty}",
        mode="lines+markers",
        line=dict(color=clr),
        yaxis="y1"
    ))

    # y2: service (billions)
    fig.add_trace(go.Bar(
        x=s_service["Year"], y=s_service["service_billion"],
        name=f"{right_name} in {pretty}",
        marker_color=clr, opacity=0.5,
        yaxis="y2"
    ))


In [5]:
conn, queries = connect_and_load_queries()

# Run queries
Q_SERVICE = queries[159]   # service output (y2)
Q_NEW   = queries[161]   # new-vehicle shares (y1)

Database scenarios: Current-Policies-Med, Enhanced-Ambition-Med, Current-Policies-High, Current-Policies-Low, Enhanced-Ambition-High, Enhanced-Ambition-Low


In [7]:
df_service = run_service_query(conn, Q_SERVICE, scenarios=SCENARIOS)
df_new = run_service_query(conn, Q_NEW, scenarios=SCENARIOS)

In [8]:
df_service_agg = aggregate_service(df_service, SERVICE_SECTORS_ALL, min_year=2020)
df_service_agg

,scenario,Year,Units,service,service_billion
0,Current-Policies-Med,2020,million pass-km,3981.640000,3.981640
1,Current-Policies-Med,2020,million ton-km,2047.490000,2.047490
2,Current-Policies-Med,2025,million pass-km,118319.081400,118.319081
3,Current-Policies-Med,2025,million ton-km,14905.312056,14.905312
4,Current-Policies-Med,2030,million pass-km,176669.373900,176.669374
5,Current-Policies-Med,2030,million ton-km,24563.995000,24.563995
6,Current-Policies-Med,2035,million pass-km,218898.589000,218.898589
7,Current-Policies-Med,2035,million ton-km,35854.847944,35.854848
8,Enhanced-Ambition-Med,2020,million pass-km,3981.640000,3.981640
9,Enhanced-Ambition-Med,2020,million ton-km,2047.490000,2.047490


In [9]:
df_share = df_new[(df_new['subsector'].isin(SERVICE_SECTORS_ALL)) & (df_new['Year'] >= 2020)].copy()
df_share['ZEV'] = df_share['technology'].apply(lambda t: 1 if t in ['BEV', 'FCEV'] else 0)
df_share

,Units,scenario,region,sector,subsector,technology,Year,value,ZEV
76,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2020,1885.17,1
77,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2025,72083.40,1
78,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2030,100685.00,1
79,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2035,116046.00,1
80,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,FCEV,2020,355.20,1
...,...,...,...,...,...,...,...,...,...
526,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2035,38632.40,1
531,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,Liquids,2020,64642.90,0
532,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,Liquids,2025,70354.30,0
533,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,Liquids,2030,63972.90,0


In [10]:
serShare = (df_share[df_share['ZEV'] == 1].groupby(['scenario', 'Units', 'Year'])['value'].sum() / df_share.groupby(['scenario', 'Units', 'Year'])['value'].sum()) * 100
serShare.reset_index()

,scenario,Units,Year,value
0,Current-Policies-Med,million pass-km,2020,0.671926
1,Current-Policies-Med,million pass-km,2025,17.471520
2,Current-Policies-Med,million pass-km,2030,22.494592
3,Current-Policies-Med,million pass-km,2035,25.913647
4,Current-Policies-Med,million ton-km,2020,3.070142
5,Current-Policies-Med,million ton-km,2025,16.021919
6,Current-Policies-Med,million ton-km,2030,14.288961
7,Current-Policies-Med,million ton-km,2035,18.510007
8,Enhanced-Ambition-Med,million pass-km,2020,0.671926
9,Enhanced-Ambition-Med,million pass-km,2025,17.471520


In [14]:
# Build figures
fig_passenger = go.Figure()
for scen in SCENARIOS:
    add_dual_series_from_split(
        fig_passenger,
        df_share=serShare.reset_index().rename(columns={'value': 'share_percent'}),
        df_service=df_service_agg,
        units_filter=UNITS_PASSENGER,
        scenario=scen,
        left_name="New Sales",
        right_name="ZEV Service"
    )
base_layout(fig_passenger, y2_title="billion pass–km", width=620, height=460, legend_pos="h")
# keep your y1 range/legend tweaks...

fig_passenger.update_layout(
    yaxis=dict(
        title=dict(text="New Sales Share (%)", font=dict(color="black")),
        range=[0, 80],  # <- yaxis1 범위 지정
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        dtick=10, mirror=True
    ),
    yaxis2=dict(
        title=dict(text="billion pass–km", font=dict(color="black")),
        tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
        overlaying="y", side="right",
        showline=True, linecolor="black", linewidth=1
    ),
    legend=dict(
        x=0.02,       # 왼쪽 여백 (0=완전 왼쪽)
        y=0.98,       # 위쪽 여백 (1=완전 위)
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1,
        orientation="v"  # 세로 정렬
    )
)

pio.write_image(fig_passenger, "./fig/pass.png", width=620, height=460, scale=2)
fig_passenger

In [13]:
fig_freight = go.Figure()
for scen in SCENARIOS:
    add_dual_series_from_split(
        fig_freight,
        df_share=serShare.reset_index().rename(columns={'value': 'share_percent'}),
        df_service=df_service_agg,
        units_filter=UNITS_FREIGHT,
        scenario=scen,
        left_name="New Sales",
        right_name="ZEV Service"
    )
base_layout(fig_freight, y2_title="billion ton–km", width=620, height=460, legend_pos="v")
fig_freight.update_layout(
    yaxis=dict(
        title=dict(text="New Sales Share (%)", font=dict(color="black")),
        range=[0, 80],  # <- yaxis1 범위 지정
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        dtick=10, mirror=True
    ),
    yaxis2=dict(
        title=dict(text="billion pass–km", font=dict(color="black")),
        tickcolor="black", ticklen=6, tickwidth=2, ticks="outside",
        overlaying="y", side="right",
        showline=True, linecolor="black", linewidth=1
    ),
    legend=dict(
        x=0.02,       # 왼쪽 여백 (0=완전 왼쪽)
        y=0.98,       # 위쪽 여백 (1=완전 위)
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1,
        orientation="v"  # 세로 정렬
    )
)
pio.write_image(fig_freight, "./fig/freight.png", width=620, height=460, scale=2)
fig_freight

In [25]:
dfZev = df_service[(df_service['technology'].isin(['FCEV', 'BEV'])) & (df_service['subsector'].isin(SERVICE_SECTORS_ALL)) & (df_service['Year'] >= 2020)].copy()
dfZev

,Units,scenario,region,sector,subsector,technology,Year,value
76,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2020,1885.170000
77,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2025,72083.400000
78,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2030,100685.000000
79,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2035,116046.000000
80,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,FCEV,2020,355.200000
...,...,...,...,...,...,...,...,...
522,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,BEV,2035,31574.380000
523,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2020,0.790000
524,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2025,11251.472056
525,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2030,34397.835000


In [29]:
def get_vehicle_num(row):
    pass_km_per_veh_car = 33 * 365 * 1.26
    pass_km_per_veh_bus = 70241290694 / 85165
    ton_km_per_veh = 47 * 365 * 4.2
    sector = row['subsector']
    value = row['value'] * 1e6
    if sector in ['Car', 'Large Car and Truck']:
        return value / pass_km_per_veh_car
    elif sector == 'Bus':
        return value / pass_km_per_veh_bus
    elif sector == 'Medium truck':
        return value / ton_km_per_veh

In [30]:
dfZev['# Vehicles'] = dfZev.apply(get_vehicle_num, axis=1)
dfZev

,Units,scenario,region,sector,subsector,technology,Year,value,# Vehicles
76,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2020,1885.170000,2285.699785
77,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2025,72083.400000,87398.490266
78,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2030,100685.000000,122076.885836
79,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,BEV,2035,116046.000000,140701.537406
80,million pass-km,Current-Policies-Med,South Korea,trn_pass_road,Bus,FCEV,2020,355.200000,430.667029
...,...,...,...,...,...,...,...,...,...
522,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,BEV,2035,31574.380000,438222.647847
523,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2020,0.790000,10.964456
524,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2025,11251.472056,156159.832008
525,million ton-km,Enhanced-Ambition-Med,South Korea,trn_freight_road,Medium truck,FCEV,2030,34397.835000,477409.543240


In [53]:
dfVeh = (dfZev.groupby(['scenario', 'Year'])['# Vehicles'].sum() / 1e6).reset_index() 
dfVeh

,scenario,Year,# Vehicles
0,Current-Policies-Med,2020,0.145867
1,Current-Policies-Med,2025,1.185179
2,Current-Policies-Med,2030,2.965174
3,Current-Policies-Med,2035,4.728543
4,Enhanced-Ambition-Med,2020,0.145867
5,Enhanced-Ambition-Med,2025,1.185179
6,Enhanced-Ambition-Med,2030,4.649435
7,Enhanced-Ambition-Med,2035,9.978175


In [74]:
# Plotly Figure 생성
fig = go.Figure()

# 시나리오별 꺾은선 그래프 추가
scenarios = list(set(dfVeh["scenario"]))
colors = {"Current-Policies-Med": "#636EFA", "Enhanced-Ambition-Med": "#00CC96"}
names = {"Current-Policies-Med": "Current Policies", "Enhanced-Ambition-Med": "Enhanced Ambition"}

for scenario in scenarios:

    mask = [s == scenario for s in dfVeh["scenario"]]
    fig.add_trace(go.Scatter(
        x=[y for y, m in zip(dfVeh["Year"], mask) if m],
        y=[v for v, m in zip(dfVeh["# Vehicles"], mask) if m],
        mode='lines+markers',
        name=names[scenario],
        line=dict(color=colors[scenario], width=3)
    ))

# base_layout 스타일 적용 (단일 y축 버전)
fig.update_layout(
    legend=dict(
        x=0.5, y=-0.2, xanchor="center", orientation="h"
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=450,
    width=600,
    font=dict(size=16),
    barmode="group",
    xaxis=dict(
        title="Year",
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        mirror=True,
        # tickformat=",",
    ),
    yaxis=dict(
        title=dict(text="Number of ZEVs", font=dict(color="black")),
        showticklabels=True,
        ticks="outside",
        ticklen=6, tickwidth=2, tickcolor="black",
        showline=True, linecolor="black", linewidth=1,
        mirror=True
    )
)

fig.update_layout(
    yaxis=dict(
        title=dict(text="# of ZEVs (mil. vehicles)", font=dict(color="black")),
        range=[0, 10],  # <- yaxis1 범위 지정
        showticklabels=True,
        tickformat=",",
    ),

    legend=dict(
        x=0.02,       # 왼쪽 여백 (0=완전 왼쪽)
        y=0.98,       # 위쪽 여백 (1=완전 위)
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="lightgray",
        borderwidth=1,
        orientation="v"  # 세로 정렬
    )
)

# 기존 꺾은선 그래프 코드 이후에 추가
fig.add_trace(go.Scatter(
    x=[2030],
    y=[4.5],
    mode='markers',
    name="Deployment Target",
    marker=dict(symbol='triangle-up', size=9, color='orange'),
))

pio.write_image(fig, "./fig/veh.png", width=620, height=460, scale=2)
fig.show()